# 05. Boost Ablation

**Question**: agent-kms applies two additive score adjustments before
threshold filtering:

- **Severity boost**: `critical +0.05 / high +0.025 / default 0`
- **Applicability penalty**: `universal 0 / conditional -0.005 / topic-specific -0.015`

And one query-side enhancement:
- **Template query expansion** (`template_expand`): appends universal-aspect
  cues to the query.

These were tuned against agent-kms's original corpus, where chunks carry
curated `severity` and `applicability` payload tags. Our public-docs
corpus has **no such curation** — every chunk uses defaults
(`severity=default`, `applicability=topic-specific`). On a flat corpus
like this, we expect:

- Severity boost → **no effect** (every chunk has the same severity).
- Applicability penalty → **uniform -0.015 shift** (every chunk is
  topic-specific, so all scores subtract the same constant — no relative
  ranking change but a shift in *how many* chunks survive the threshold).
- Query expansion → **possibly relevant** — it could pull in more chunks
  by broadening the query embedding.

We isolate each variable at the corpus-best threshold found in notebook 03.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from src.eval_utils import resolve_eval_paths

CORPUS_DIR, QUERIES_PATH, COLLECTION = resolve_eval_paths(ROOT)
os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = COLLECTION
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(CORPUS_DIR)

from agent_kms.retrieve import retrieve

from src.eval_utils import (
    f1_at_threshold,
    gold_set,
    load_queries,
    precision_at_threshold,
    recall_at_threshold,
)

sns.set_theme(style="whitegrid")


In [ ]:
# Best threshold from notebook 03 (read from results CSV so this notebook
# stays in sync if 03 is re-run with different settings).
sweep_csv = ROOT / "results" / "threshold_sweep.csv"
T = 0.83
if sweep_csv.exists():
    s = pd.read_csv(sweep_csv)
    T = float(s.loc[s["f1"].idxmax(), "threshold"])
print(f"Using best threshold T = {T} (from notebook 03 sweep)")

queries = load_queries(QUERIES_PATH)
in_domain = [q for q in queries if q.get("gold")]
adversarial = [q for q in queries if not q.get("gold")]
print(f"in-domain {len(in_domain)} | adversarial {len(adversarial)}")


In [ ]:
# Conditions — each disables one component (or all of them) at the same T.
# default_boost / default_app are agent-kms's defaults.
DEFAULT_SEV  = {"critical": 0.05, "high": 0.025, "default": 0.0}
DEFAULT_APP  = {"universal": 0.0, "conditional": -0.005, "topic-specific": -0.015}

conditions = {
    "all on (default)":       dict(expand=True,  severity_boost=DEFAULT_SEV, applicability_boost=DEFAULT_APP),
    "no severity boost":      dict(expand=True,  severity_boost={},          applicability_boost=DEFAULT_APP),
    "no applicability penalty": dict(expand=True, severity_boost=DEFAULT_SEV, applicability_boost={}),
    "no query expand":        dict(expand=False, severity_boost=DEFAULT_SEV, applicability_boost=DEFAULT_APP),
    "vanilla (none of the above)": dict(expand=False, severity_boost={},     applicability_boost={}),
}

def eval_one(cond_kwargs):
    rows_in = []
    for q in in_domain:
        r = retrieve(q["query"], score_threshold=T, **cond_kwargs)
        g = gold_set(q)
        rows_in.append({
            "precision": precision_at_threshold(r, g),
            "recall":    recall_at_threshold(r, g),
            "f1":        f1_at_threshold(r, g),
            "num_returned": len(r),
        })
    rows_adv = []
    for q in adversarial:
        r = retrieve(q["query"], score_threshold=T, **cond_kwargs)
        rows_adv.append({"num_returned": len(r)})
    df_in = pd.DataFrame(rows_in)
    df_adv = pd.DataFrame(rows_adv)
    return dict(
        precision=df_in["precision"].mean(),
        recall=df_in["recall"].mean(),
        f1=df_in["f1"].mean(),
        avg_returned=df_in["num_returned"].mean(),
        adv_empty_rate=(df_adv["num_returned"] == 0).mean(),
    )

rows = []
for name, kw in conditions.items():
    res = eval_one(kw)
    res["condition"] = name
    rows.append(res)

summary = pd.DataFrame(rows).set_index("condition")[
    ["precision", "recall", "f1", "avg_returned", "adv_empty_rate"]
]
print(f"\nAblation results at T={T}:")
print(summary.round(3).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
summary[["precision", "recall", "f1"]].plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_title(f"Boost ablation — quality metrics at T={T}")
ax.set_ylabel("score (mean over in-domain queries)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(loc="upper right")

ax = axes[1]
ax.bar(summary.index, summary["avg_returned"], color="#9467bd")
ax.set_title(f"Boost ablation — volume at T={T}")
ax.set_ylabel("avg returned chunks (in-domain)")
ax.set_xticklabels(summary.index, rotation=20, ha="right")
for i, v in enumerate(summary["avg_returned"]):
    ax.text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_boost_ablation.png", dpi=140, bbox_inches="tight")
plt.show()

summary.to_csv(ROOT / "results" / "boost_ablation.csv")


## Findings

Empirical results at the corpus-best threshold T=0.81 (multilingual-e5-base):

| condition                    | P     | R     | F1    | avg ret. | adv empty |
|------------------------------|-------|-------|-------|----------|-----------|
| all on (default)             | 0.159 | 0.649 | 0.201 | 13.2     | 0.75      |
| no severity boost            | 0.159 | 0.649 | 0.201 | 13.2     | 0.75      |
| no applicability penalty     | 0.067 | 0.804 | 0.111 | 40.8     | 0.00      |
| no query expand              | 0.082 | 0.929 | 0.133 | 33.6     | 0.50      |
| vanilla (none of the above)  | 0.030 | 1.000 | 0.058 | 69.0     | 0.00      |

- **Severity boost is exactly a no-op on this corpus** — disabling it
  leaves every metric unchanged to 3 decimal places. Reason: every chunk
  ingested under the default `severity=default` tag, so the boost dict's
  per-tag adjustment evaluates to 0 for all of them.
- **Applicability penalty (-0.015 for `topic-specific` chunks) is the
  single most impactful filter.** Removing it triples `avg_returned`
  (13 → 41) and halves F1 (0.20 → 0.11). It effectively acts as a hidden
  threshold tightening of 0.015.
- **Query expansion suppresses returned-count too.** Disabling expansion
  raises recall to 0.93 (more chunks pass the threshold) but lets
  enough noise through to halve F1. Adversarial empty-rate also drops
  from 0.75 to 0.50 — i.e. expansion *also* helps protect against
  out-of-domain queries.
- **Defaults are the best F1 point at this T** for this corpus, but
  they hit that F1 by lowering the *effective* threshold via the
  applicability penalty, not by surfacing extra signal. The same F1
  could be reached with a fresh threshold sweep on
  `retrieve(score_threshold=T, applicability_boost={})`.

**Implication for agent-kms users**: the documented `0.93` default
implicitly bakes in a `-0.015` shift for uncurated corpora. If you
plan to skip the curation step, you should either:
1. Tag every chunk's `applicability=universal` so the penalty becomes 0, or
2. Pass `applicability_boost={}` and lower your threshold by ~0.015 to
   compensate.

Both are clearer than relying on the hidden shift.
